# SOLETE — Astronomical Feature Engineering

This notebook develops deterministic astronomical features for the
selected 5-minute SOLETE forecasting dataset.

The purpose is to introduce physically meaningful information describing
the position and cycles of the Sun and Moon while keeping these features
separate from both:

- conventional Gregorian calendar features, and
- Vedic/Panchang calendar features.

This separation supports the planned feature-ablation experiments:

- F0 — historical/autoregressive baseline;
- F1 — + Gregorian features;
- F2 — + solar astronomical features;
- F3 — + lunar astronomical features;
- F4 — + Vedic/Panchang features.

## Methodological Principle

Astronomical variables used for modelling will be computed independently
from:

1. timestamp;
2. geographic location;
3. an explicit astronomical calculation method.

The unusual dataset-provided `Azimuth[deg]` and `Elevation[deg]` columns
will not be used as the primary astronomical model features.

Where useful, dataset variables may be retained only for diagnostic
comparison.

All astronomical quantities considered as future-known features must be
deterministically computable for the forecast timestamp and must not
depend on future measured weather or power observations.

In [1]:
# ============================================================
# CELL 1 — SITE AND TIMESTAMP ASSUMPTIONS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Astronomical calculations depend on both:
#
#     WHERE  -> geographic coordinates
#     WHEN   -> precise timestamp / time standard
#
# These assumptions are therefore defined explicitly before
# any solar or lunar position is calculated.
#
# No astronomical features are created in this cell.


from pathlib import Path
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. PROJECT ROOT
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


print("Project root:")
print(PROJECT_ROOT)


# ============================================================
# 2. SOLETE / DTU SITE COORDINATES
# ============================================================
#
# SOLETE measurements were collected at the SYSLAB facility
# of DTU in Denmark.
#
# Coordinate convention:
#
#     latitude:
#         north  -> positive
#         south  -> negative
#
#     longitude:
#         east   -> positive
#         west   -> negative
#
# These coordinates will be the geographic reference for
# astronomical calculations.

SITE_NAME = "DTU SYSLAB, Denmark"

SITE_LATITUDE_DEG = 55.6867
SITE_LONGITUDE_DEG = 12.0985


# ------------------------------------------------------------
# Altitude
# ------------------------------------------------------------
#
# We have not yet established a verified site altitude from
# the dataset documentation.
#
# Therefore, we deliberately do NOT invent one.
#
# If the astronomical method later requires altitude, we will
# verify it from an appropriate source before using it.

SITE_ALTITUDE_M = None


# ============================================================
# 3. TIME REFERENCE
# ============================================================
#
# SOLETE timestamps are represented in UTC.
#
# UTC will remain the canonical astronomical input time
# because it:
#
#     - is unambiguous;
#     - avoids daylight-saving-time gaps and repetitions;
#     - preserves the continuous 5-minute dataset timeline.
#
# Europe/Copenhagen is recorded separately as the site's
# civil timezone.
#
# Civil local time may later be useful for interpretation,
# but it will NOT replace UTC as the canonical timestamp.

DATA_TIMEZONE = "UTC"

SITE_CIVIL_TIMEZONE = "Europe/Copenhagen"


# ============================================================
# 4. 5-MINUTE TIMESTAMP INTERPRETATION
# ============================================================
#
# The SOLETE 5-minute dataset contains processed / averaged
# measurements.
#
# At this stage we do NOT assume without verification whether
# each timestamp should astronomically represent:
#
#     A. the beginning of the 5-minute interval,
#
#           t
#
#     or
#
#     B. the midpoint of the interval,
#
#           t + 2.5 minutes
#
# This matters because the Sun's position changes continuously.
#
# We will initially preserve the dataset timestamp exactly and
# later test timestamp-vs-midpoint interpretations during
# astronomical validation.

SAMPLING_INTERVAL_MINUTES = 5

TIMESTAMP_INTERPRETATION = (
    "To be validated: timestamp versus 2.5-minute interval midpoint"
)


# ============================================================
# 5. DISPLAY ASSUMPTIONS
# ============================================================

site_assumptions = pd.Series(
    {
        "Site": SITE_NAME,
        "Latitude [deg]": SITE_LATITUDE_DEG,
        "Longitude [deg]": SITE_LONGITUDE_DEG,
        "Altitude [m]": SITE_ALTITUDE_M,
        "Dataset timezone": DATA_TIMEZONE,
        "Civil timezone": SITE_CIVIL_TIMEZONE,
        "Sampling interval [min]": SAMPLING_INTERVAL_MINUTES,
        "Timestamp interpretation": TIMESTAMP_INTERPRETATION,
    }
)


print("\nAstronomical site/time assumptions:\n")

site_assumptions

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

Astronomical site/time assumptions:



Site                                                      DTU SYSLAB, Denmark
Latitude [deg]                                                        55.6867
Longitude [deg]                                                       12.0985
Altitude [m]                                                             None
Dataset timezone                                                          UTC
Civil timezone                                              Europe/Copenhagen
Sampling interval [min]                                                     5
Timestamp interpretation    To be validated: timestamp versus 2.5-minute i...
dtype: object

In [2]:
# ============================================================
# CELL 2 — LOAD 5-MINUTE DATA AND INSPECT TIMEZONE / DST
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The SOLETE dataset uses UTC timestamps.
#
# DTU/SYSLAB is physically located in Denmark, whose civil
# timezone is:
#
#     Europe/Copenhagen
#
# Denmark observes daylight-saving time (DST).
#
# Therefore:
#
#     UTC timeline:
#         continuous and unambiguous
#
#     local civil-time timeline:
#         contains a spring clock jump
#         and an autumn repeated hour
#
# This distinction is important because:
#
#   - astronomical calculations should use a precise and
#     unambiguous time reference;
#
#   - local civil time may still be useful for interpretation
#     or later calendar-related analysis.
#
# No astronomical features are calculated in this cell.


# ============================================================
# 1. LOAD THE SELECTED 5-MINUTE DATASET
# ============================================================

from src.data_loader import load_solete


dataset_path = (
    PROJECT_ROOT
    / "solete_dataset"
    / "SOLETE_Pombo_5min.h5"
)


assert dataset_path.exists(), (
    f"Dataset not found: {dataset_path}"
)


df_raw = load_solete(
    dataset_path
)


print("Dataset shape:")
print(df_raw.shape)

print("\nUTC date range:")
print(
    df_raw.index.min(),
    "to",
    df_raw.index.max()
)


# ============================================================
# 2. VERIFY UTC TIMESTAMP PROPERTIES
# ============================================================

print("\nDataset timezone:")
print(df_raw.index.tz)

print("\nUTC chronological order:")
print(
    df_raw.index.is_monotonic_increasing
)

print("\nUTC duplicate timestamps:")
print(
    df_raw.index.duplicated().sum()
)


# ============================================================
# 3. CONVERT UTC TO COPENHAGEN CIVIL TIME
# ============================================================
#
# tz_convert changes only the timezone representation.
#
# It does NOT change the physical instant represented by
# each timestamp.

local_index = df_raw.index.tz_convert(
    SITE_CIVIL_TIMEZONE
)


print("\nFirst UTC timestamp:")
print(
    df_raw.index[0]
)

print("\nSame instant in Copenhagen civil time:")
print(
    local_index[0]
)


print("\nLast UTC timestamp:")
print(
    df_raw.index[-1]
)

print("\nSame instant in Copenhagen civil time:")
print(
    local_index[-1]
)


# ============================================================
# 4. VERIFY THAT TIMEZONE-AWARE LOCAL TIMESTAMPS REMAIN UNIQUE
# ============================================================
#
# Even during the autumn DST transition, timezone-aware
# timestamps remain distinguishable because their UTC offsets
# differ.

print("\nTimezone-aware local timestamps duplicated:")
print(
    local_index.duplicated().sum()
)


# ============================================================
# 5. EXAMINE LOCAL WALL-CLOCK TIME
# ============================================================
#
# Removing timezone information creates naive local
# wall-clock timestamps.
#
# During the autumn DST transition, one local clock hour
# occurs twice.
#
# This demonstrates why naive local timestamps should NOT
# replace UTC as our canonical time index.

local_naive_index = (
    local_index.tz_localize(None)
)


print("\nRepeated naive local wall-clock timestamps:")
print(
    local_naive_index.duplicated().sum()
)


# ============================================================
# 6. IDENTIFY UTC-OFFSET CHANGES
# ============================================================
#
# Copenhagen switches between:
#
#     UTC+1  -> standard time
#     UTC+2  -> daylight-saving time
#
# We detect the exact timestamps where that offset changes
# within the SOLETE observation period.

utc_offset_hours = pd.Series(
    [
        timestamp.utcoffset().total_seconds() / 3600
        for timestamp in local_index
    ],
    index=df_raw.index,
    name="utc_offset_hours",
)


offset_change_mask = (
    utc_offset_hours
    != utc_offset_hours.shift(1)
)

# The first row has no previous observation and is therefore
# excluded from the list of actual transitions.
offset_change_mask.iloc[0] = False


dst_transitions = pd.DataFrame(
    {
        "UTC_timestamp":
            df_raw.index[
                offset_change_mask
            ],

        "Copenhagen_timestamp":
            local_index[
                offset_change_mask
            ],

        "previous_UTC_offset_hours":
            utc_offset_hours
            .shift(1)
            .loc[
                offset_change_mask
            ]
            .to_numpy(),

        "new_UTC_offset_hours":
            utc_offset_hours
            .loc[
                offset_change_mask
            ]
            .to_numpy(),
    }
)


print("\nDST / UTC-offset transitions found:")
print(
    len(dst_transitions)
)

print("\nTransitions:")

dst_transitions

Dataset shape:
(131617, 11)

UTC date range:
2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00

Dataset timezone:
UTC

UTC chronological order:
True

UTC duplicate timestamps:
0

First UTC timestamp:
2018-06-01 00:00:00+00:00

Same instant in Copenhagen civil time:
2018-06-01 02:00:00+02:00

Last UTC timestamp:
2019-09-01 00:00:00+00:00

Same instant in Copenhagen civil time:
2019-09-01 02:00:00+02:00

Timezone-aware local timestamps duplicated:
0

Repeated naive local wall-clock timestamps:
12

DST / UTC-offset transitions found:
2

Transitions:


,UTC_timestamp,Copenhagen_timestamp,previous_UTC_offset_hours,new_UTC_offset_hours
0,2018-10-28 01:00:00+00:00,2018-10-28 02:00:00+01:00,2.0,1.0
1,2019-03-31 01:00:00+00:00,2019-03-31 03:00:00+02:00,1.0,2.0


## Astronomical Computation Strategy

Two complementary astronomical tools will be used because the solar and
lunar feature groups have different requirements.

### Solar Astronomy — F2

Solar-position features will be calculated using **pvlib** and its
implementation of the NREL Solar Position Algorithm (SPA).

The primary solar quantities considered will include:

- geometric solar elevation;
- solar zenith;
- solar azimuth;
- deterministic daylight information.

Geometric solar position will be preferred initially so that the
features do not depend on future meteorological measurements such as
temperature or atmospheric pressure.

Both the original dataset timestamp and the 2.5-minute interval midpoint
will be evaluated before fixing the timestamp convention.

### Lunar Astronomy — F3

Lunar astronomical quantities will be calculated using **Skyfield** with
a documented JPL planetary/lunar ephemeris.

The lunar feature family will initially remain astronomical rather than
calendar-based and may include:

- Sun-Moon angular separation / lunar phase angle;
- cyclic representation of lunar phase;
- other lunar positional quantities if justified.

### Vedic / Panchang Features — F4

Vedic/Panchang quantities will be constructed only after the underlying
solar and lunar astronomy has been independently validated.

Where practical, these quantities will be derived explicitly rather than
obtained from an opaque calendar-conversion package.

This is important because the research must distinguish between:

1. conventional solar astronomical information;
2. conventional lunar astronomical information;
3. the additional representation introduced by the Vedic lunisolar
   calendar.

Choices such as sidereal reference system and ayanamsha will be stated
explicitly before features such as Nakshatra or Yoga are calculated.

In [3]:
# ============================================================
# CELL 3 — VERIFY ASTRONOMY SOFTWARE ENVIRONMENT
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Astronomical calculations can depend on the implementation
# and library version used.
#
# Therefore, we explicitly record the versions of:
#
#     pvlib
#     skyfield
#
# used in this research.
#
# This improves reproducibility and makes the astronomical
# methodology easier to document later in the thesis.


import pvlib
import skyfield


print("pvlib version:")
print(pvlib.__version__)

print("\nSkyfield version:")
print(skyfield.__version__)


# ------------------------------------------------------------
# BASIC IMPORT VERIFICATION
# ------------------------------------------------------------
#
# pvlib will initially be used for solar-position calculations.
#
# Skyfield will later be used for lunar astronomy and the
# astronomical quantities underlying the Vedic/Panchang
# feature family.

from pvlib import solarposition

from skyfield.api import load


print("\nSolar-position module imported successfully:")
print(solarposition is not None)

print("\nSkyfield loader imported successfully:")
print(load is not None)

pvlib version:
0.15.2

Skyfield version:
1.55

Solar-position module imported successfully:
True

Skyfield loader imported successfully:
True


## Solar-Position Sanity Check

Before generating solar features for the complete dataset, a small set of
representative timestamps is evaluated using the NREL Solar Position
Algorithm implemented by `pvlib`.

The purpose is to verify that:

- solar elevation is low or negative during nighttime;
- solar elevation increases through the morning;
- the Sun reaches a high elevation near local solar noon;
- solar azimuth progresses smoothly through the day;
- the calculated quantities are physically plausible for Denmark.

This validation is performed before choosing the final timestamp
interpretation for the 5-minute averaged observations.

In [4]:
# ============================================================
# CELL 4 — SOLAR-POSITION SANITY TEST
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Before computing solar position for all 131,617 timestamps,
# we test the calculation on a few representative times.
#
# We use a summer date because the daily solar cycle is easy
# to inspect at Denmark's latitude.
#
# The calculations use:
#
#     pvlib
#     NREL Solar Position Algorithm (SPA)
#     DTU/SYSLAB latitude and longitude
#     timezone-aware UTC timestamps
#
# No SOLETE measurements are used in the astronomical
# calculation.


# ============================================================
# 1. SELECT REPRESENTATIVE UTC TIMESTAMPS
# ============================================================
#
# June 21 is close to the Northern Hemisphere summer
# solstice.
#
# We sample several times across the day rather than the
# complete 5-minute series.

solar_test_times = pd.DatetimeIndex(
    [
        "2018-06-21 00:00:00+00:00",
        "2018-06-21 04:00:00+00:00",
        "2018-06-21 08:00:00+00:00",
        "2018-06-21 11:00:00+00:00",
        "2018-06-21 14:00:00+00:00",
        "2018-06-21 18:00:00+00:00",
        "2018-06-21 22:00:00+00:00",
    ]
)


# ============================================================
# 2. CALCULATE SOLAR POSITION
# ============================================================
#
# nrel_numpy uses pvlib's NumPy implementation of the
# NREL Solar Position Algorithm.
#
# For the first validation we focus primarily on the
# geometric quantities:
#
#     elevation
#     zenith
#     azimuth
#
# Geometric elevation is particularly useful because it does
# not require us to treat future measured meteorological
# conditions as forecast-known information.

solar_test_position = (
    pvlib.solarposition.get_solarposition(
        time=solar_test_times,
        latitude=SITE_LATITUDE_DEG,
        longitude=SITE_LONGITUDE_DEG,
        method="nrel_numpy",
    )
)


# ============================================================
# 3. ADD LOCAL CIVIL TIME FOR INTERPRETATION
# ============================================================
#
# Local time is shown only to make the daily pattern easier
# to interpret.
#
# The astronomical calculation itself uses the timezone-aware
# UTC timestamps.

solar_test_results = pd.DataFrame(
    {
        "Copenhagen_time":
            solar_test_times.tz_convert(
                SITE_CIVIL_TIMEZONE
            ),

        "solar_elevation_deg":
            solar_test_position["elevation"],

        "solar_zenith_deg":
            solar_test_position["zenith"],

        "solar_azimuth_deg":
            solar_test_position["azimuth"],

        "apparent_elevation_deg":
            solar_test_position["apparent_elevation"],
    },
    index=solar_test_times,
)


# Give the index an explicit name.
solar_test_results.index.name = "UTC_timestamp"


# ============================================================
# 4. DISPLAY RESULTS
# ============================================================

print("Solar-position sanity check:")
print(
    f"Site: {SITE_NAME}"
)

print(
    f"Latitude: {SITE_LATITUDE_DEG}°"
)

print(
    f"Longitude: {SITE_LONGITUDE_DEG}°"
)

solar_test_results

Solar-position sanity check:
Site: DTU SYSLAB, Denmark
Latitude: 55.6867°
Longitude: 12.0985°


,Copenhagen_time,solar_elevation_deg,solar_zenith_deg,solar_azimuth_deg,apparent_elevation_deg
UTC_timestamp,,,,,
2018-06-21 00:00:00+00:00,2018-06-21 02:00:00+02:00,-10.256969,100.256969,10.879131,-10.256969
2018-06-21 04:00:00+00:00,2018-06-21 06:00:00+02:00,9.543095,80.456905,62.028775,9.636784
2018-06-21 08:00:00+00:00,2018-06-21 10:00:00+02:00,42.242926,47.757074,112.186347,42.261433
2018-06-21 11:00:00+00:00,2018-06-21 13:00:00+02:00,57.652708,32.347292,174.254918,57.663365
2018-06-21 14:00:00+00:00,2018-06-21 16:00:00+02:00,45.640708,44.359292,240.704256,45.657151
2018-06-21 18:00:00+00:00,2018-06-21 20:00:00+02:00,12.951042,77.048958,292.762551,13.021466
2018-06-21 22:00:00+00:00,2018-06-22 00:00:00+02:00,-9.347010,99.347010,342.958375,-9.347010


## Validation of the 5-Minute Astronomical Timestamp Convention

The SOLETE 5-minute measurements represent processed interval data, while
each row is labelled by a single timestamp.

Two astronomical interpretations are therefore compared:

1. solar position evaluated exactly at the recorded timestamp `t`;
2. solar position evaluated at `t + 2.5 minutes`, the midpoint of the
   nominal 5-minute interval.

The dataset-provided `Elevation[deg]` and `Azimuth[deg]` variables are
used only as diagnostic references.

They are not treated as astronomical ground truth because previous EDA
showed that their representation is clipped/nonstandard, especially
outside daylight periods.

For this reason:

- elevation comparison is restricted to observations where the
  dataset-provided elevation is positive;
- azimuth differences are calculated circularly;
- the comparison is used only to help choose the temporal convention for
  independently computed astronomical features.

In [5]:
# ============================================================
# CELL 5 — COMPARE TIMESTAMP VS 5-MINUTE INTERVAL MIDPOINT
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We compare solar position computed at:
#
#     A. the recorded SOLETE timestamp
#
#     B. timestamp + 2.5 minutes
#
# against the dataset-provided solar-position variables.
#
# IMPORTANT:
#
# The SOLETE Azimuth/Elevation columns are used only as
# diagnostic references because their representation was
# previously found to be nonstandard / clipped.
#
# No final astronomical feature convention is selected until
# we inspect the results.


# ============================================================
# 1. DEFINE THE TWO TIME INTERPRETATIONS
# ============================================================

solar_times_timestamp = (
    df_raw.index
)

solar_times_midpoint = (
    df_raw.index
    + pd.Timedelta(minutes=2.5)
)


# ============================================================
# 2. COMPUTE SOLAR POSITION FOR BOTH INTERPRETATIONS
# ============================================================

solar_at_timestamp = (
    pvlib.solarposition.get_solarposition(
        time=solar_times_timestamp,
        latitude=SITE_LATITUDE_DEG,
        longitude=SITE_LONGITUDE_DEG,
        method="nrel_numpy",
    )
)


solar_at_midpoint = (
    pvlib.solarposition.get_solarposition(
        time=solar_times_midpoint,
        latitude=SITE_LATITUDE_DEG,
        longitude=SITE_LONGITUDE_DEG,
        method="nrel_numpy",
    )
)


# ------------------------------------------------------------
# Re-align midpoint results to the original SOLETE timestamps
# so that comparisons use the same row labels.
# ------------------------------------------------------------

solar_at_midpoint.index = df_raw.index


# ============================================================
# 3. DEFINE VALID DIAGNOSTIC COMPARISON MASK
# ============================================================
#
# Dataset Elevation=0 does not reliably mean astronomical
# elevation is exactly zero.
#
# Therefore, compare only rows where the dataset provides a
# positive elevation value.

solar_reference_mask = (
    df_raw["Elevation[deg]"] > 0
)


print("Rows with positive dataset-provided elevation:")
print(
    solar_reference_mask.sum()
)


# ============================================================
# 4. ELEVATION COMPARISON
# ============================================================

dataset_elevation = (
    df_raw.loc[
        solar_reference_mask,
        "Elevation[deg]"
    ]
)


timestamp_elevation_error = (
    solar_at_timestamp.loc[
        solar_reference_mask,
        "elevation"
    ]
    -
    dataset_elevation
).abs()


midpoint_elevation_error = (
    solar_at_midpoint.loc[
        solar_reference_mask,
        "elevation"
    ]
    -
    dataset_elevation
).abs()


print("\nElevation MAE — recorded timestamp:")
print(
    timestamp_elevation_error.mean()
)

print("\nElevation MAE — +2.5 minute midpoint:")
print(
    midpoint_elevation_error.mean()
)


print("\nElevation median absolute error — recorded timestamp:")
print(
    timestamp_elevation_error.median()
)

print("\nElevation median absolute error — +2.5 minute midpoint:")
print(
    midpoint_elevation_error.median()
)


# ============================================================
# 5. CIRCULAR AZIMUTH COMPARISON
# ============================================================
#
# Azimuth is circular.
#
# Example:
#
#     359° and 1°
#
# differ by only 2°, not 358°.
#
# We therefore calculate the smallest angular separation.

dataset_azimuth = (
    df_raw.loc[
        solar_reference_mask,
        "Azimuth[deg]"
    ]
)


def circular_angle_difference_deg(angle_a, angle_b):
    """
    Smallest absolute angular difference between two angles.

    Result range:
        0 to 180 degrees
    """

    return np.abs(
        (
            angle_a
            - angle_b
            + 180
        )
        % 360
        - 180
    )


timestamp_azimuth_error = (
    circular_angle_difference_deg(
        solar_at_timestamp.loc[
            solar_reference_mask,
            "azimuth"
        ],
        dataset_azimuth,
    )
)


midpoint_azimuth_error = (
    circular_angle_difference_deg(
        solar_at_midpoint.loc[
            solar_reference_mask,
            "azimuth"
        ],
        dataset_azimuth,
    )
)


print("\nAzimuth circular MAE — recorded timestamp:")
print(
    timestamp_azimuth_error.mean()
)

print("\nAzimuth circular MAE — +2.5 minute midpoint:")
print(
    midpoint_azimuth_error.mean()
)


print("\nAzimuth circular median error — recorded timestamp:")
print(
    timestamp_azimuth_error.median()
)

print("\nAzimuth circular median error — +2.5 minute midpoint:")
print(
    midpoint_azimuth_error.median()
)


# ============================================================
# 6. COMPACT SUMMARY TABLE
# ============================================================

timestamp_alignment_comparison = pd.DataFrame(
    {
        "metric": [
            "Elevation MAE [deg]",
            "Elevation median error [deg]",
            "Azimuth circular MAE [deg]",
            "Azimuth circular median error [deg]",
        ],

        "recorded_timestamp": [
            timestamp_elevation_error.mean(),
            timestamp_elevation_error.median(),
            timestamp_azimuth_error.mean(),
            timestamp_azimuth_error.median(),
        ],

        "interval_midpoint_+2.5min": [
            midpoint_elevation_error.mean(),
            midpoint_elevation_error.median(),
            midpoint_azimuth_error.mean(),
            midpoint_azimuth_error.median(),
        ],
    }
)


print("\nTimestamp-alignment comparison:")

timestamp_alignment_comparison

Rows with positive dataset-provided elevation:
70572

Elevation MAE — recorded timestamp:
5.601444186420647

Elevation MAE — +2.5 minute midpoint:
5.843722201174966

Elevation median absolute error — recorded timestamp:
6.423407027977339

Elevation median absolute error — +2.5 minute midpoint:
6.7054790170429595

Azimuth circular MAE — recorded timestamp:
161.72475934928391

Azimuth circular MAE — +2.5 minute midpoint:
161.09522284599277

Azimuth circular median error — recorded timestamp:
162.36723720878757

Azimuth circular median error — +2.5 minute midpoint:
161.79305277672677

Timestamp-alignment comparison:


,metric,recorded_timestamp,interval_midpoint_+2.5min
0,Elevation MAE [deg],5.601444,5.843722
1,Elevation median error [deg],6.423407,6.705479
2,Azimuth circular MAE [deg],161.724759,161.095223
3,Azimuth circular median error [deg],162.367237,161.793053


### Preliminary Timestamp-Alignment Finding

The dataset-provided solar variables cannot yet be used directly to
select the 5-minute timestamp convention.

For positive dataset elevation values:

- solar elevation calculated at the recorded timestamp gives a slightly
  lower error than the +2.5-minute midpoint;
- however, both elevation comparisons still show several degrees of
  disagreement.

More importantly, the direct azimuth comparison produces errors of
approximately 161–162 degrees.

Such a large difference cannot plausibly be explained by a 2.5-minute
timestamp shift. It instead suggests that the SOLETE `Azimuth[deg]`
variable may use a different angular reference or sign convention from
the independently calculated pvlib azimuth.

Therefore, the dataset azimuth convention will be investigated before
the timestamp-versus-midpoint decision is finalized.

In [6]:
# ============================================================
# CELL 6 — DIAGNOSE SOLETE AZIMUTH CONVENTION
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 5 found an approximately 160-degree disagreement
# between:
#
#     SOLETE Azimuth[deg]
#
# and:
#
#     pvlib solar azimuth
#
# Such a large discrepancy is unlikely to come from the
# 2.5-minute timestamp question.
#
# Instead, the source data may use a different azimuth
# convention.
#
# We therefore test several common angular transformations
# WITHOUT modifying the source Azimuth[deg] column.
#
# Comparison is restricted to rows where the SOLETE
# elevation is positive.


# ============================================================
# 1. GET SOURCE AZIMUTH FOR VALID DAYLIGHT REFERENCE ROWS
# ============================================================

source_azimuth = (
    df_raw.loc[
        solar_reference_mask,
        "Azimuth[deg]"
    ]
)


print("SOLETE azimuth range on positive-elevation rows:")
print(
    source_azimuth.min(),
    "to",
    source_azimuth.max()
)


# ============================================================
# 2. CREATE POSSIBLE AZIMUTH-CONVENTION TRANSFORMATIONS
# ============================================================
#
# These are diagnostic candidates only.
#
# Candidate A:
#     source stored directly on conventional 0–360 scale
#
# Candidate B:
#     source uses 0° at South:
#
#         converted = source + 180°
#
# Candidate C:
#     reverse angular direction around North
#
# Candidate D:
#     reverse direction around South
#
# All candidates are wrapped into [0°, 360°).

azimuth_candidates = pd.DataFrame(
    {
        "as_stored":
            source_azimuth % 360,

        "plus_180":
            (source_azimuth + 180) % 360,

        "reverse_north":
            (-source_azimuth) % 360,

        "reverse_south":
            (180 - source_azimuth) % 360,
    },
    index=source_azimuth.index,
)


# ============================================================
# 3. COMPARE EACH CONVENTION WITH PVLIB
# ============================================================
#
# We evaluate both:
#
#     recorded timestamp
#     +2.5-minute midpoint
#
# so the azimuth convention and timestamp question remain
# separate.

pvlib_azimuth_timestamp = (
    solar_at_timestamp.loc[
        solar_reference_mask,
        "azimuth"
    ]
)

pvlib_azimuth_midpoint = (
    solar_at_midpoint.loc[
        solar_reference_mask,
        "azimuth"
    ]
)


comparison_rows = []

for candidate_name in azimuth_candidates.columns:

    candidate = (
        azimuth_candidates[candidate_name]
    )

    timestamp_error = (
        circular_angle_difference_deg(
            pvlib_azimuth_timestamp,
            candidate,
        )
    )

    midpoint_error = (
        circular_angle_difference_deg(
            pvlib_azimuth_midpoint,
            candidate,
        )
    )

    comparison_rows.append(
        {
            "candidate_convention":
                candidate_name,

            "timestamp_MAE_deg":
                timestamp_error.mean(),

            "timestamp_median_deg":
                timestamp_error.median(),

            "midpoint_MAE_deg":
                midpoint_error.mean(),

            "midpoint_median_deg":
                midpoint_error.median(),
        }
    )


azimuth_convention_comparison = pd.DataFrame(
    comparison_rows
)


# ============================================================
# 4. DISPLAY RESULTS
# ============================================================

print("\nAzimuth convention comparison:")

azimuth_convention_comparison.sort_values(
    "timestamp_MAE_deg"
)

SOLETE azimuth range on positive-elevation rows:
-142.760009765625 to 136.16342981974287

Azimuth convention comparison:


,candidate_convention,timestamp_MAE_deg,timestamp_median_deg,midpoint_MAE_deg,midpoint_median_deg
1,plus_180,18.275241,17.632763,18.904777,18.206947
2,reverse_north,78.413510,71.810044,78.442734,71.856241
3,reverse_south,101.586490,108.189956,101.557266,108.143759
0,as_stored,161.724759,162.367237,161.095223,161.793053


### SOLETE Azimuth-Convention Finding

The SOLETE `Azimuth[deg]` variable is not expressed using the same
angular convention as pvlib.

On rows with positive dataset-provided solar elevation:

- using the SOLETE values directly produces an azimuth MAE of
  approximately **161.7°**;
- adding 180° and wrapping into `[0°, 360°)` reduces the MAE to
  approximately **18.3°**.

The source azimuth itself ranges approximately from `-143°` to `+136°`,
which is consistent with a signed solar-azimuth convention referenced
approximately to South.

For diagnostic comparison, the likely conversion is therefore:

`pvlib-style azimuth ≈ (SOLETE azimuth + 180°) mod 360°`

However, the remaining approximately 18° azimuth error and 5–6°
elevation error are still much larger than expected from a 2.5-minute
sampling-interval shift.

This suggests that the dataset-provided solar-position variables may
also involve a different time-reference convention.

The next diagnostic tests whether a UTC/local-civil-time offset can
explain this residual disagreement.

In [7]:
# ============================================================
# CELL 7 — DIAGNOSE POSSIBLE SOLAR-POSITION TIME OFFSET
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 6 established that SOLETE azimuth is most consistent
# with a signed South-referenced convention:
#
#     pvlib-style azimuth
#         ≈ (SOLETE azimuth + 180°) % 360
#
# Even after this conversion, however, substantial residual
# differences remain:
#
#     elevation MAE ≈ 5–6°
#     azimuth MAE   ≈ 18°
#
# These errors are much too large to be explained by a
# 2.5-minute interval-midpoint shift.
#
# One possible explanation is a time-reference mismatch in
# the dataset-provided solar-position columns.
#
# We therefore compare the SOLETE solar variables with pvlib
# calculations evaluated under several candidate time shifts.
#
# This is an OFFLINE diagnostic only.
# The original timestamps are never modified.


# ============================================================
# 1. USE ONLY ROWS WITH POSITIVE SOLETE ELEVATION
# ============================================================

reference_times = (
    df_raw.index[
        solar_reference_mask
    ]
)


source_elevation = (
    df_raw.loc[
        solar_reference_mask,
        "Elevation[deg]"
    ]
    .to_numpy()
)


# ------------------------------------------------------------
# Convert SOLETE's likely signed South-referenced azimuth
# into the north-referenced 0–360° convention used by pvlib.
# ------------------------------------------------------------

source_azimuth_converted = (
    (
        df_raw.loc[
            solar_reference_mask,
            "Azimuth[deg]"
        ]
        + 180
    )
    % 360
).to_numpy()


# ============================================================
# 2. GET COPENHAGEN UTC OFFSET FOR EACH REFERENCE TIMESTAMP
# ============================================================
#
# Depending on season:
#
#     standard time -> UTC+1
#     daylight time -> UTC+2
#
# We already calculated utc_offset_hours in Cell 2.

reference_utc_offsets = (
    utc_offset_hours.loc[
        solar_reference_mask
    ]
    .to_numpy()
)


# ============================================================
# 3. DEFINE CANDIDATE TIME INTERPRETATIONS
# ============================================================
#
# Fixed shifts test simple clock offsets.
#
# The two civil-offset candidates test a DST-aware shift:
#
#     + Copenhagen UTC offset
#     - Copenhagen UTC offset
#
# If one DST-aware candidate becomes dramatically better,
# that would suggest local/UTC clock handling in the source
# solar-position variables.

candidate_time_sets = {
    "-2 hours":
        reference_times
        - pd.Timedelta(hours=2),

    "-1 hour":
        reference_times
        - pd.Timedelta(hours=1),

    "recorded UTC":
        reference_times,

    "+1 hour":
        reference_times
        + pd.Timedelta(hours=1),

    "+2 hours":
        reference_times
        + pd.Timedelta(hours=2),

    "- Copenhagen civil offset":
        reference_times
        - pd.to_timedelta(
            reference_utc_offsets,
            unit="h"
        ),

    "+ Copenhagen civil offset":
        reference_times
        + pd.to_timedelta(
            reference_utc_offsets,
            unit="h"
        ),
}


# ============================================================
# 4. CALCULATE ERROR FOR EACH TIME INTERPRETATION
# ============================================================

time_offset_rows = []


for candidate_name, candidate_times in candidate_time_sets.items():

    candidate_position = (
        pvlib.solarposition.get_solarposition(
            time=candidate_times,
            latitude=SITE_LATITUDE_DEG,
            longitude=SITE_LONGITUDE_DEG,
            method="nrel_numpy",
        )
    )


    # --------------------------------------------------------
    # Elevation error
    # --------------------------------------------------------

    calculated_elevation = (
        candidate_position["elevation"]
        .to_numpy()
    )

    elevation_error = np.abs(
        calculated_elevation
        -
        source_elevation
    )


    # --------------------------------------------------------
    # Circular azimuth error
    # --------------------------------------------------------

    calculated_azimuth = (
        candidate_position["azimuth"]
        .to_numpy()
    )

    azimuth_error = np.abs(
        (
            calculated_azimuth
            -
            source_azimuth_converted
            +
            180
        )
        % 360
        -
        180
    )


    # --------------------------------------------------------
    # Store summary
    # --------------------------------------------------------

    time_offset_rows.append(
        {
            "candidate_time_reference":
                candidate_name,

            "elevation_MAE_deg":
                elevation_error.mean(),

            "elevation_median_error_deg":
                np.median(elevation_error),

            "azimuth_MAE_deg":
                azimuth_error.mean(),

            "azimuth_median_error_deg":
                np.median(azimuth_error),
        }
    )


# ============================================================
# 5. DISPLAY COMPARISON
# ============================================================

solar_time_reference_comparison = (
    pd.DataFrame(
        time_offset_rows
    )
)


print(
    "Solar-position time-reference comparison:\n"
)

solar_time_reference_comparison.sort_values(
    "elevation_MAE_deg"
)

Solar-position time-reference comparison:



,candidate_time_reference,elevation_MAE_deg,elevation_median_error_deg,azimuth_MAE_deg,azimuth_median_error_deg
1,-1 hour,0.621777,0.619059,3.181119,3.764249
5,- Copenhagen civil offset,5.294592,6.711052,10.602830,10.153165
2,recorded UTC,5.601444,6.423407,18.275241,17.632763
0,-2 hours,6.210165,7.113309,11.943532,10.507121
3,+1 hour,11.334282,13.074191,33.396578,31.539393
6,+ Copenhagen civil offset,15.759471,17.356455,45.387235,42.459875
4,+2 hours,16.791065,19.298332,48.552705,45.551839


### Solar-Position Source-Column Diagnostic Outcome

The remaining discrepancy in the dataset-provided solar-position
variables is strongly associated with an approximately fixed one-hour
time shift.

When the independently calculated pvlib solar position is evaluated one
hour before the recorded SOLETE timestamp:

- elevation MAE decreases from approximately **5.60°** to **0.62°**;
- circular azimuth MAE decreases from approximately **18.28°** to
  **3.18°** after accounting for the source azimuth convention.

A DST-aware Copenhagen civil-time correction does not reproduce this
improvement, suggesting that the pattern is more consistent with a
fixed source-processing or timestamp-alignment convention than with
ordinary daylight-saving-time conversion.

This diagnostic does **not** imply that the SOLETE measurement timeline
itself should be shifted.

The supplied `Azimuth[deg]` and `Elevation[deg]` variables will therefore
remain diagnostic variables and will not be used as the primary
astronomical forecasting features.

Subsequent solar features will be computed independently from the
verified UTC timestamp and DTU/SYSLAB geographic coordinates.

In [8]:
# ============================================================
# CELL 8 — VALIDATE MIDPOINT REPRESENTATION OF 5-MINUTE DATA
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Previous cross-resolution analysis established that the
# provided 5-minute SOLETE data closely follow averaging of
# the 1-minute series.
#
# Therefore, a row labelled:
#
#     t
#
# represents measurements over approximately:
#
#     t, t+1 min, t+2 min, t+3 min, t+4 min
#
# For deterministic astronomical features, we want a single
# solar position that represents this interval.
#
# A natural choice is the interval midpoint:
#
#     t + 2.5 minutes
#
# Instead of relying on the unusual SOLETE Elevation/Azimuth
# columns, we independently test how closely midpoint solar
# position approximates the mean astronomical state across
# each five-minute interval.
#
# No measured SOLETE variable is used in this validation.


# ============================================================
# 1. CREATE COMPLETE 1-MINUTE ASTRONOMICAL TIME GRID
# ============================================================

astronomy_1min_times = pd.date_range(
    start=df_raw.index.min(),
    end=df_raw.index.max(),
    freq="1min",
    tz="UTC",
)


print("Number of 1-minute astronomical timestamps:")
print(
    len(astronomy_1min_times)
)


# ============================================================
# 2. CALCULATE SOLAR POSITION AT 1-MINUTE RESOLUTION
# ============================================================
#
# We use geometric solar position.
#
# Elevation can be averaged linearly.
#
# Azimuth is circular and therefore requires a circular mean.

solar_1min = (
    pvlib.solarposition.get_solarposition(
        time=astronomy_1min_times,
        latitude=SITE_LATITUDE_DEG,
        longitude=SITE_LONGITUDE_DEG,
        method="nrel_numpy",
    )
)


# ============================================================
# 3. CALCULATE 5-MINUTE MEAN ELEVATION
# ============================================================

elevation_5min_mean = (
    solar_1min["elevation"]
    .resample("5min")
    .mean()
    .reindex(df_raw.index)
)


# ============================================================
# 4. CALCULATE 5-MINUTE CIRCULAR-MEAN AZIMUTH
# ============================================================
#
# Ordinary arithmetic means are inappropriate for angles.
#
# Example:
#
#     mean(359°, 1°)
#
# should be approximately 0°, not 180°.
#
# We therefore average sine/cosine components and convert
# the result back to degrees.

azimuth_1min_rad = np.deg2rad(
    solar_1min["azimuth"]
)

azimuth_sin_1min = pd.Series(
    np.sin(azimuth_1min_rad),
    index=astronomy_1min_times,
)

azimuth_cos_1min = pd.Series(
    np.cos(azimuth_1min_rad),
    index=astronomy_1min_times,
)


azimuth_sin_5min = (
    azimuth_sin_1min
    .resample("5min")
    .mean()
    .reindex(df_raw.index)
)

azimuth_cos_5min = (
    azimuth_cos_1min
    .resample("5min")
    .mean()
    .reindex(df_raw.index)
)


azimuth_5min_circular_mean = (
    np.rad2deg(
        np.arctan2(
            azimuth_sin_5min,
            azimuth_cos_5min,
        )
    )
    % 360
)


# ============================================================
# 5. GET SOLAR POSITION AT THE 2.5-MINUTE MIDPOINT
# ============================================================

midpoint_elevation = (
    solar_at_midpoint["elevation"]
)

midpoint_azimuth = (
    solar_at_midpoint["azimuth"]
)


# ============================================================
# 6. COMPARE MIDPOINT WITH INTERVAL-AVERAGED ASTRONOMY
# ============================================================

midpoint_elevation_error = (
    midpoint_elevation
    -
    elevation_5min_mean
).abs()


midpoint_azimuth_error = (
    circular_angle_difference_deg(
        midpoint_azimuth,
        azimuth_5min_circular_mean,
    )
)


# ------------------------------------------------------------
# The final dataset endpoint contains only one 1-minute
# timestamp rather than a complete five-minute interval.
#
# Exclude incomplete interval bins from this methodological
# comparison.
# ------------------------------------------------------------

interval_counts = (
    pd.Series(
        1,
        index=astronomy_1min_times
    )
    .resample("5min")
    .count()
    .reindex(df_raw.index)
)

complete_interval_mask = (
    interval_counts == 5
)


# ============================================================
# 7. DISPLAY VALIDATION RESULTS
# ============================================================

print("Complete 5-minute intervals:")
print(
    complete_interval_mask.sum()
)

print("\nIncomplete intervals:")
print(
    (~complete_interval_mask).sum()
)


print(
    "\nMidpoint vs 5-minute mean elevation MAE [deg]:"
)
print(
    midpoint_elevation_error.loc[
        complete_interval_mask
    ].mean()
)


print(
    "\nMidpoint vs 5-minute mean elevation "
    "maximum error [deg]:"
)
print(
    midpoint_elevation_error.loc[
        complete_interval_mask
    ].max()
)


print(
    "\nMidpoint vs circular-mean azimuth MAE [deg]:"
)
print(
    midpoint_azimuth_error.loc[
        complete_interval_mask
    ].mean()
)


print(
    "\nMidpoint vs circular-mean azimuth "
    "maximum error [deg]:"
)
print(
    midpoint_azimuth_error.loc[
        complete_interval_mask
    ].max()
)

Number of 1-minute astronomical timestamps:
658081
Complete 5-minute intervals:
131616

Incomplete intervals:
1

Midpoint vs 5-minute mean elevation MAE [deg]:
0.04765926457536803

Midpoint vs 5-minute mean elevation maximum error [deg]:
0.07059475482722233

Midpoint vs circular-mean azimuth MAE [deg]:
0.12500084497943645

Midpoint vs circular-mean azimuth maximum error [deg]:
0.21488486355193004


### Five-Minute Astronomical Representative-Time Check

The 2.5-minute interval midpoint provides a very close approximation to
the independently calculated mean solar state across each 5-minute
period.

The observed errors are small:

- elevation MAE: approximately **0.048°**;
- maximum elevation error: approximately **0.071°**;
- circular azimuth MAE: approximately **0.125°**;
- maximum circular azimuth error: approximately **0.215°**.

One additional distinction must nevertheless be considered.

The 1-minute reconstruction used five samples located at:

`t`, `t+1`, `t+2`, `t+3`, and `t+4` minutes.

The central timestamp of these five discrete samples is therefore
`t+2 minutes`, whereas the geometric midpoint of the continuous
5-minute interval `[t, t+5 min)` is `t+2.5 minutes`.

Because this distinction is small but methodologically relevant, the
recorded timestamp, discrete-sample centre, and continuous-interval
midpoint will be compared once before fixing the final astronomical
reference-time convention.

In [9]:
# ============================================================
# CELL 9 — SELECT 5-MINUTE ASTRONOMICAL REPRESENTATIVE TIME
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 8 showed that the +2.5-minute midpoint closely
# represents the mean astronomical state of a 5-minute bin.
#
# However, the reconstructed 1-minute mean uses observations
# at:
#
#     t, t+1, t+2, t+3, t+4
#
# whose discrete temporal centre is t+2 minutes.
#
# We therefore compare three possible single-time
# representatives:
#
#     A. t          -> recorded timestamp
#     B. t + 2 min  -> centre of the five 1-minute samples
#     C. t + 2.5min -> midpoint of the continuous 5-minute bin
#
# The reference remains our independently calculated
# 1-minute astronomical mean, NOT the unusual SOLETE
# Elevation/Azimuth columns.


# ============================================================
# 1. DEFINE CANDIDATE REPRESENTATIVE TIMES
# ============================================================

representative_time_candidates = {
    "recorded_timestamp_t":
        df_raw.index,

    "discrete_center_t+2min":
        df_raw.index
        + pd.Timedelta(minutes=2),

    "interval_midpoint_t+2.5min":
        df_raw.index
        + pd.Timedelta(minutes=2.5),
}


# ============================================================
# 2. CALCULATE SOLAR POSITION FOR EACH CANDIDATE
# ============================================================

representative_time_rows = []


for candidate_name, candidate_times in (
    representative_time_candidates.items()
):

    candidate_position = (
        pvlib.solarposition.get_solarposition(
            time=candidate_times,
            latitude=SITE_LATITUDE_DEG,
            longitude=SITE_LONGITUDE_DEG,
            method="nrel_numpy",
        )
    )


    # Re-align the calculated results with the original
    # SOLETE row timestamps.
    candidate_position.index = df_raw.index


    # --------------------------------------------------------
    # Elevation error relative to the independent
    # five-sample 1-minute mean.
    # --------------------------------------------------------

    elevation_error = (
        candidate_position["elevation"]
        -
        elevation_5min_mean
    ).abs()


    # --------------------------------------------------------
    # Circular azimuth error relative to the independent
    # five-sample circular mean.
    # --------------------------------------------------------

    azimuth_error = (
        circular_angle_difference_deg(
            candidate_position["azimuth"],
            azimuth_5min_circular_mean,
        )
    )


    # --------------------------------------------------------
    # Store only complete 5-minute intervals.
    # --------------------------------------------------------

    representative_time_rows.append(
        {
            "representative_time":
                candidate_name,

            "elevation_MAE_deg":
                elevation_error.loc[
                    complete_interval_mask
                ].mean(),

            "elevation_max_error_deg":
                elevation_error.loc[
                    complete_interval_mask
                ].max(),

            "azimuth_MAE_deg":
                azimuth_error.loc[
                    complete_interval_mask
                ].mean(),

            "azimuth_max_error_deg":
                azimuth_error.loc[
                    complete_interval_mask
                ].max(),
        }
    )


# ============================================================
# 3. DISPLAY COMPARISON
# ============================================================

representative_time_comparison = pd.DataFrame(
    representative_time_rows
)


print(
    "Five-minute astronomical representative-time comparison:\n"
)

representative_time_comparison

Five-minute astronomical representative-time comparison:



,representative_time,elevation_MAE_deg,elevation_max_error_deg,azimuth_MAE_deg,azimuth_max_error_deg
0,recorded_timestamp_t,0.190628,0.282372,0.500004,0.859457
1,discrete_center_t+2min,0.000391,0.001057,0.000327,0.001197
2,interval_midpoint_t+2.5min,0.047659,0.070595,0.125001,0.214885


### Final Astronomical Representative-Time Decision

The representative time for deterministic astronomical features is fixed
at:

**recorded SOLETE timestamp + 2 minutes**

This choice is based on the independently established 1-minute-to-5-minute
processing lineage.

The provided 5-minute observations closely follow the mean of five
1-minute samples located at:

`t`, `t+1`, `t+2`, `t+3`, and `t+4` minutes.

The temporal centre of these five discrete samples is therefore
`t+2 minutes`.

Independent solar-position validation confirms this choice:

- elevation MAE relative to the five-sample astronomical mean:
  approximately **0.00039°**;
- maximum elevation error:
  approximately **0.00106°**;
- circular azimuth MAE:
  approximately **0.00033°**;
- maximum circular azimuth error:
  approximately **0.00120°**.

These errors are substantially smaller than those obtained using either
the recorded timestamp or the continuous-interval midpoint at
`t+2.5 minutes`.

Therefore, all subsequently engineered astronomical features for the
5-minute SOLETE dataset will be evaluated at `t+2 minutes`.

This convention is specific to the observed SOLETE aggregation lineage
and should not be generalized automatically to other datasets.

In [10]:
# ============================================================
# CELL 10 — CREATE FINAL F2 SOLAR ASTRONOMICAL FEATURES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cells 4–9 established and validated the astronomical
# calculation methodology.
#
# Final conventions:
#
#   Site:
#       DTU/SYSLAB, Denmark
#
#   Canonical dataset time:
#       UTC
#
#   Astronomical representative time:
#       recorded timestamp + 2 minutes
#
#   Solar-position algorithm:
#       pvlib implementation of NREL SPA
#
# We now generate the deterministic F2 solar feature family.
#
# The dataset-provided Azimuth/Elevation columns are NOT used
# to construct these features.


# ============================================================
# 1. DEFINE FINAL ASTRONOMICAL REFERENCE TIMES
# ============================================================

astro_reference_times = (
    df_raw.index
    + pd.Timedelta(minutes=2)
)


# ============================================================
# 2. CALCULATE INDEPENDENT SOLAR POSITION
# ============================================================

solar_position_final = (
    pvlib.solarposition.get_solarposition(
        time=astro_reference_times,
        latitude=SITE_LATITUDE_DEG,
        longitude=SITE_LONGITUDE_DEG,
        method="nrel_numpy",
    )
)


# ------------------------------------------------------------
# Re-align the result with the original SOLETE timestamps.
#
# The feature at row t represents astronomy at t+2 minutes,
# but the dataset row itself remains indexed by t.
# ------------------------------------------------------------

solar_position_final.index = (
    df_raw.index
)


# ============================================================
# 3. CREATE A SEPARATE ASTRONOMICAL FEATURE TABLE
# ============================================================
#
# Keeping astronomy in a separate DataFrame makes the feature
# families easier to control during later ablation experiments.

df_solar_features = pd.DataFrame(
    index=df_raw.index
)


# ------------------------------------------------------------
# Solar elevation
# ------------------------------------------------------------
#
# Geometric elevation:
#
#     > 0°  -> Sun geometrically above horizon
#     = 0°  -> geometric horizon
#     < 0°  -> Sun below horizon

df_solar_features["solar_elevation_deg"] = (
    solar_position_final["elevation"]
)


# ------------------------------------------------------------
# Cosine of solar zenith
# ------------------------------------------------------------
#
# Since:
#
#     zenith = 90° - elevation
#
# cos(zenith) provides a compact geometric measure of the
# Sun's orientation relative to the local vertical.
#
# It is often physically relevant to solar-energy processes.

solar_zenith_rad = np.deg2rad(
    solar_position_final["zenith"]
)

df_solar_features["solar_cos_zenith"] = (
    np.cos(solar_zenith_rad)
)


# ------------------------------------------------------------
# Circular solar-azimuth representation
# ------------------------------------------------------------
#
# pvlib azimuth convention:
#
#     North =   0°
#     East  =  90°
#     South = 180°
#     West  = 270°
#
# Raw azimuth is circular and therefore represented using
# sine and cosine components.

solar_azimuth_rad = np.deg2rad(
    solar_position_final["azimuth"]
)

df_solar_features["solar_azimuth_sin"] = (
    np.sin(solar_azimuth_rad)
)

df_solar_features["solar_azimuth_cos"] = (
    np.cos(solar_azimuth_rad)
)


# ------------------------------------------------------------
# Deterministic daylight indicator
# ------------------------------------------------------------
#
# This is based on geometric solar elevation rather than
# measured irradiance.
#
# It is therefore known in advance for future timestamps.

df_solar_features["solar_daylight"] = (
    df_solar_features["solar_elevation_deg"]
    > 0
).astype("int8")


# ============================================================
# 4. VERIFY FEATURE TABLE
# ============================================================

print("Solar feature table shape:")
print(
    df_solar_features.shape
)


print("\nMissing values:")
print(
    df_solar_features.isna().sum()
)


print("\nFeature ranges:")

for column in df_solar_features.columns:

    print(
        f"{column:22s}: "
        f"{df_solar_features[column].min(): .6f} "
        f"to "
        f"{df_solar_features[column].max(): .6f}"
    )


# ============================================================
# 5. VERIFY AZIMUTH CIRCULAR NORM
# ============================================================

solar_azimuth_norm = (
    df_solar_features["solar_azimuth_sin"] ** 2
    +
    df_solar_features["solar_azimuth_cos"] ** 2
)


print(
    "\nMaximum solar-azimuth circular norm error:"
)

print(
    np.abs(
        solar_azimuth_norm - 1
    ).max()
)


# ============================================================
# 6. DAYLIGHT SUMMARY
# ============================================================

print("\nDaylight observations:")
print(
    df_solar_features["solar_daylight"].sum()
)

print("\nNight / below-horizon observations:")
print(
    (
        df_solar_features["solar_daylight"] == 0
    ).sum()
)


# ============================================================
# 7. DISPLAY SAMPLE
# ============================================================

print("\nSample F2 solar astronomical features:")

df_solar_features.head(12)

Solar feature table shape:
(131617, 5)

Missing values:
solar_elevation_deg    0
solar_cos_zenith       0
solar_azimuth_sin      0
solar_azimuth_cos      0
solar_daylight         0
dtype: int64

Feature ranges:
solar_elevation_deg   : -57.747676 to  57.746609
solar_cos_zenith      : -0.845706 to  0.845696
solar_azimuth_sin     : -1.000000 to  1.000000
solar_azimuth_cos     : -1.000000 to  1.000000
solar_daylight        :  0.000000 to  1.000000

Maximum solar-azimuth circular norm error:
2.220446049250313e-16

Daylight observations:
70745

Night / below-horizon observations:
60872

Sample F2 solar astronomical features:


,solar_elevation_deg,solar_cos_zenith,solar_azimuth_sin,solar_azimuth_cos,solar_daylight
Timestamp,,,,,
2018-06-01 00:00:00+00:00,-11.497634,-0.199327,0.215420,0.976522,0
2018-06-01 00:05:00+00:00,-11.338361,-0.196603,0.235330,0.971916,0
2018-06-01 00:10:00+00:00,-11.165112,-0.193637,0.255095,0.966916,0
2018-06-01 00:15:00+00:00,-10.977992,-0.190432,0.274705,0.961529,0
2018-06-01 00:20:00+00:00,-10.777115,-0.186989,0.294148,0.955760,0
2018-06-01 00:25:00+00:00,-10.562601,-0.183310,0.313415,0.949616,0
2018-06-01 00:30:00+00:00,-10.334577,-0.179396,0.332497,0.943104,0
2018-06-01 00:35:00+00:00,-10.093179,-0.175250,0.351383,0.936232,0
2018-06-01 00:40:00+00:00,-9.838547,-0.170872,0.370065,0.929006,0


## Physical Validation of the F2 Solar Features

The independently calculated solar features should exhibit physically
reasonable relationships with measured solar irradiance and PV output.

This validation compares deterministic solar geometry with:

- `GHI[kW1m2]`;
- `POA Irr[kW1m2]`;
- `P_Solar[kW]`.

These comparisons are diagnostic only.

Measured irradiance and PV power at the same timestamp are not treated
as deterministic future-known forecasting inputs. Their use here is
solely to verify that the independently calculated astronomical features
behave consistently with the observed solar-energy cycle.

Correlation is also interpreted cautiously because nighttime zeros and
seasonal structure can strengthen same-timestamp relationships.

In [11]:
# ============================================================
# CELL 11 — PHYSICAL VALIDATION OF F2 SOLAR FEATURES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We now verify that the independently calculated solar
# astronomy behaves consistently with the observed solar
# energy variables.
#
# We examine:
#
#     solar elevation
#     positive / negative geometric daylight
#
# against:
#
#     GHI
#     POA irradiance
#     solar PV power
#
# IMPORTANT:
#
# These are SAME-TIMESTAMP diagnostic comparisons.
#
# They do NOT imply that future measured GHI, POA, or
# P_Solar values are available as forecasting inputs.


# ============================================================
# 1. COMBINE ONLY THE VARIABLES NEEDED FOR VALIDATION
# ============================================================

solar_validation = pd.concat(
    [
        df_solar_features[
            [
                "solar_elevation_deg",
                "solar_cos_zenith",
                "solar_daylight",
            ]
        ],

        df_raw[
            [
                "GHI[kW1m2]",
                "POA Irr[kW1m2]",
                "P_Solar[kW]",
            ]
        ],
    ],
    axis=1,
)


# ============================================================
# 2. SAME-TIMESTAMP CORRELATIONS
# ============================================================
#
# These are descriptive physical-validation statistics,
# not forecasting-performance estimates.

solar_validation_correlations = (
    solar_validation[
        [
            "solar_elevation_deg",
            "solar_cos_zenith",
            "GHI[kW1m2]",
            "POA Irr[kW1m2]",
            "P_Solar[kW]",
        ]
    ]
    .corr()
)


print(
    "Correlations with independently calculated "
    "solar elevation:\n"
)

print(
    solar_validation_correlations.loc[
        "solar_elevation_deg",
        [
            "GHI[kW1m2]",
            "POA Irr[kW1m2]",
            "P_Solar[kW]",
        ],
    ]
)


print(
    "\nCorrelations with solar cos(zenith):\n"
)

print(
    solar_validation_correlations.loc[
        "solar_cos_zenith",
        [
            "GHI[kW1m2]",
            "POA Irr[kW1m2]",
            "P_Solar[kW]",
        ],
    ]
)


# ============================================================
# 3. CHECK NIGHTTIME OBSERVATIONS
# ============================================================
#
# Geometric night:
#
#     solar elevation <= 0°
#
# Solar power or irradiance may occasionally remain slightly
# positive close to sunrise/sunset because:
#
#     - measurements are averaged over a 5-minute interval;
#     - atmospheric refraction is not represented by geometric
#       elevation;
#     - diffuse irradiance may exist near the geometric horizon;
#     - measurement / processing conventions may differ.
#
# Therefore, nonzero values are inspected rather than
# automatically labelled erroneous.

geometric_night_mask = (
    df_solar_features["solar_daylight"] == 0
)


print("\nGeometric-night observations:")
print(
    geometric_night_mask.sum()
)


print("\nNight observations with P_Solar > 0:")
print(
    (
        geometric_night_mask
        &
        (df_raw["P_Solar[kW]"] > 0)
    ).sum()
)


print("\nNight observations with GHI > 0:")
print(
    (
        geometric_night_mask
        &
        (df_raw["GHI[kW1m2]"] > 0)
    ).sum()
)


print("\nNight observations with POA irradiance > 0:")
print(
    (
        geometric_night_mask
        &
        (df_raw["POA Irr[kW1m2]"] > 0)
    ).sum()
)


# ============================================================
# 4. COMPARE DAYLIGHT AND NIGHTTIME SOLAR POWER
# ============================================================

solar_power_by_daylight = (
    solar_validation
    .groupby("solar_daylight")[
        [
            "GHI[kW1m2]",
            "POA Irr[kW1m2]",
            "P_Solar[kW]",
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "max",
        ]
    )
)


print(
    "\nSolar-energy measurements by geometric daylight state:"
)

solar_power_by_daylight

Correlations with independently calculated solar elevation:

GHI[kW1m2]        0.774692
POA Irr[kW1m2]    0.692185
P_Solar[kW]       0.688781
Name: solar_elevation_deg, dtype: float64

Correlations with solar cos(zenith):

GHI[kW1m2]        0.769514
POA Irr[kW1m2]    0.688214
P_Solar[kW]       0.685898
Name: solar_cos_zenith, dtype: float64

Geometric-night observations:
60872

Night observations with P_Solar > 0:
6671

Night observations with GHI > 0:
2584

Night observations with POA irradiance > 0:
10878

Solar-energy measurements by geometric daylight state:


GHI[kW1m2]                    POA Irr[kW1m2]            \
                     mean   median       max           mean    median   
solar_daylight                                                          
0                0.000079  0.00000  0.029160       0.000167  0.000000   
1                0.261105  0.18382  1.092413       0.268471  0.141473   

                         P_Solar[kW]                     
                     max        mean   median       max  
solar_daylight                                           
0               0.023840    0.001283  0.00000  0.191199  
1               1.119663    1.836976  0.98988  7.715423

### F2 Solar-Astronomy Validation Outcome

The independently calculated solar astronomical features exhibit the
expected physical relationship with measured irradiance and PV
generation.

Same-timestamp correlations with independently calculated solar
elevation are approximately:

- GHI: **0.775**
- POA irradiance: **0.692**
- PV power: **0.689**

The corresponding relationships with `cos(zenith)` are similar.

These relationships are physically reasonable but are not interpreted
as forecasting-performance estimates. Actual irradiance and PV power
also depend on atmospheric conditions, cloud cover, system behaviour,
and other factors not determined by solar geometry alone.

### Geometric Night

Some observations retain small positive GHI, POA, or PV values when the
geometric solar elevation is below zero.

However, their magnitudes are very small relative to daytime values.

For example, during geometric night:

- mean GHI is approximately **0.00008 kW/m²**;
- mean POA irradiance is approximately **0.00017 kW/m²**;
- mean PV power is approximately **0.0013 kW**;
- maximum PV power is only approximately **0.19 kW**.

These observations are therefore not automatically classified as data
errors. Small values near the geometric horizon can arise from
five-minute averaging, atmospheric/refraction effects, diffuse light,
sensor thresholds, and processing conventions.

### F2 Status

The F2 solar astronomical feature family is considered validated.

The primary independently calculated variables are:

- `solar_elevation_deg`
- `solar_cos_zenith`
- `solar_azimuth_sin`
- `solar_azimuth_cos`
- `solar_daylight`

These variables are deterministic functions of timestamp and geographic
location and can therefore be calculated for future forecast timestamps
without using future weather or power measurements.

Feature redundancy will be addressed during modelling/ablation rather
than by discarding useful deterministic representations at this stage.

In [12]:
# ============================================================
# CELL 12 — PREPARE SKYFIELD TIME SYSTEM
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# F3 introduces independently calculated lunar astronomy.
#
# Skyfield requires:
#
#     1. a time scale;
#     2. a JPL planetary/lunar ephemeris.
#
# Before downloading or loading an ephemeris file, we first
# initialize Skyfield's time system and verify that our final
# astronomical reference timestamps can be represented
# correctly.
#
# Our previously selected convention remains:
#
#     SOLETE timestamp + 2 minutes
#
# No lunar features are created in this cell.


from skyfield.api import load


# ============================================================
# 1. CREATE SKYFIELD TIME SCALE
# ============================================================

ts = load.timescale()


print("Skyfield time scale created:")
print(ts is not None)


# ============================================================
# 2. SELECT A FEW REPRESENTATIVE UTC TIMES
# ============================================================

lunar_test_times = pd.DatetimeIndex(
    [
        "2018-06-01 00:02:00+00:00",
        "2018-09-01 12:02:00+00:00",
        "2018-12-01 00:02:00+00:00",
        "2019-03-01 12:02:00+00:00",
        "2019-06-01 00:02:00+00:00",
    ]
)


# ============================================================
# 3. CONVERT PANDAS UTC TIMES TO SKYFIELD TIMES
# ============================================================
#
# Skyfield accepts timezone-aware Python datetime objects.
#
# This preserves the precise UTC instants represented by the
# pandas timestamps.

skyfield_test_times = ts.from_datetimes(
    lunar_test_times.to_pydatetime()
)


# ============================================================
# 4. VERIFY THE CONVERSION
# ============================================================

print("\nNumber of test timestamps:")
print(
    len(lunar_test_times)
)

print("\nOriginal pandas UTC timestamps:")
print(
    lunar_test_times
)

print("\nSkyfield UTC representation:")
print(
    skyfield_test_times.utc_iso()
)

Skyfield time scale created:
True

Number of test timestamps:
5

Original pandas UTC timestamps:
DatetimeIndex(['2018-06-01 00:02:00+00:00', '2018-09-01 12:02:00+00:00',
               '2018-12-01 00:02:00+00:00', '2019-03-01 12:02:00+00:00',
               '2019-06-01 00:02:00+00:00'],
              dtype='datetime64[us, UTC]', freq=None)

Skyfield UTC representation:
['2018-06-01T00:02:00Z', '2018-09-01T12:02:00Z', '2018-12-01T00:02:00Z', '2019-03-01T12:02:00Z', '2019-06-01T00:02:00Z']


## JPL Ephemeris Selection for Lunar Astronomy

Lunar astronomical quantities require a planetary/lunar ephemeris.

This study uses the JPL **DE440s** ephemeris through Skyfield.

DE440s was selected because:

- it covers the complete SOLETE study period (2018–2019);
- it is a relatively compact JPL ephemeris;
- it provides the Earth, Moon, and Sun positions required for the
  F3 lunar astronomical features;
- the exact ephemeris is explicitly documented so that later lunar and
  Panchang calculations are reproducible.

The ephemeris file is stored separately from the SOLETE measurements and
is treated as deterministic astronomical reference data.

In [13]:
# ============================================================
# CELL 13 — LOAD AND VERIFY JPL DE440s EPHEMERIS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Skyfield requires a JPL ephemeris for high-precision
# positions of the Earth, Moon, and Sun.
#
# We use:
#
#     de440s.bsp
#
# The file is stored in a dedicated project-level directory
# rather than inside the notebook directory.
#
# The first execution may download approximately 32 MB.
# Later executions will reuse the local file.


from skyfield.api import Loader


# ============================================================
# 1. CREATE PROJECT-LOCAL EPHEMERIS DIRECTORY
# ============================================================

EPHEMERIS_DIR = (
    PROJECT_ROOT
    / "ephemeris"
)

EPHEMERIS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("Ephemeris directory:")
print(
    EPHEMERIS_DIR
)


# ============================================================
# 2. CREATE A SKYFIELD LOADER FOR THIS DIRECTORY
# ============================================================

skyfield_loader = Loader(
    str(EPHEMERIS_DIR)
)


# ============================================================
# 3. LOAD JPL DE440s
# ============================================================
#
# If de440s.bsp is not already present, Skyfield will
# download it on this first execution.

eph = skyfield_loader(
    "de440s.bsp"
)


# ============================================================
# 4. VERIFY REQUIRED CELESTIAL BODIES
# ============================================================

earth = eph["Earth"]
moon = eph["Moon"]
sun = eph["Sun"]


print("\nRequired ephemeris bodies loaded:")

print(
    "Earth:",
    earth is not None
)

print(
    "Moon:",
    moon is not None
)

print(
    "Sun:",
    sun is not None
)


# ============================================================
# 5. VERIFY LOCAL EPHEMERIS FILE
# ============================================================

ephemeris_file = (
    EPHEMERIS_DIR
    / "de440s.bsp"
)


print("\nEphemeris file exists:")
print(
    ephemeris_file.exists()
)


if ephemeris_file.exists():

    print(
        "\nEphemeris file size [MB]:"
    )

    print(
        ephemeris_file.stat().st_size
        / (1024 ** 2)
    )

Ephemeris directory:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/ephemeris

Required ephemeris bodies loaded:
Earth: True
Moon: True
Sun: True

Ephemeris file exists:
True

Ephemeris file size [MB]:
31.2099609375


## Lunar Phase-Angle Sanity Check

Before creating lunar features for the complete SOLETE timeline, the
Skyfield lunar-phase calculation is evaluated at a small set of
representative timestamps.

Skyfield defines lunar phase angle as the geocentric apparent ecliptic
longitude difference:

**Moon longitude − Sun longitude**

wrapped to the interval `[0°, 360°)`.

The principal phase locations are therefore:

- `0°`   — New Moon
- `90°`  — First Quarter
- `180°` — Full Moon
- `270°` — Last Quarter

This continuous angle is particularly important for the research design
because it provides an ordinary astronomical lunar representation for
F3 before the Vedic/Panchang representation is introduced in F4.

No Panchang category is constructed at this stage.

In [14]:
# ============================================================
# CELL 14 — MOON-PHASE ANGLE SANITY CHECK
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Before calculating lunar astronomy for all 131,617 SOLETE
# rows, verify Skyfield's continuous Moon-phase angle on the
# small timestamp sample created in Cell 12.
#
# Skyfield defines:
#
#     lunar phase angle
#         = apparent geocentric ecliptic longitude of Moon
#           minus
#           apparent geocentric ecliptic longitude of Sun
#
# wrapped into:
#
#     0° <= phase < 360°
#
# Principal locations:
#
#       0° = New Moon
#      90° = First Quarter
#     180° = Full Moon
#     270° = Last Quarter
#
# This is an ASTRONOMICAL quantity.
#
# We do NOT calculate Tithi, Paksha, Nakshatra, Yoga,
# Karana, or any other Panchang quantity in this cell.


from skyfield import almanac


# ============================================================
# 1. CALCULATE CONTINUOUS LUNAR PHASE ANGLE
# ============================================================

moon_phase_test = almanac.moon_phase(
    eph,
    skyfield_test_times,
)


moon_phase_test_deg = (
    moon_phase_test.degrees
)


# ============================================================
# 2. CREATE A SMALL INSPECTION TABLE
# ============================================================

moon_phase_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            lunar_test_times,

        "moon_phase_angle_deg":
            moon_phase_test_deg,
    }
)


# ============================================================
# 3. VERIFY NUMERICAL RANGE
# ============================================================

print("Moon-phase angle range [deg]:")

print(
    moon_phase_test_results[
        "moon_phase_angle_deg"
    ].min(),
    "to",
    moon_phase_test_results[
        "moon_phase_angle_deg"
    ].max(),
)


print("\nAll phase angles inside [0, 360):")

print(
    (
        (
            moon_phase_test_results[
                "moon_phase_angle_deg"
            ]
            >= 0
        )
        &
        (
            moon_phase_test_results[
                "moon_phase_angle_deg"
            ]
            < 360
        )
    ).all()
)


# ============================================================
# 4. DISPLAY TEST RESULTS
# ============================================================

print(
    "\nLunar phase-angle sanity check:"
)

moon_phase_test_results

Moon-phase angle range [deg]:
206.80214662342783 to 330.0868649547806

All phase angles inside [0, 360):
True

Lunar phase-angle sanity check:


,UTC_timestamp,moon_phase_angle_deg
0,2018-06-01 00:02:00+00:00,206.802147
1,2018-09-01 12:02:00+00:00,249.834414
2,2018-12-01 00:02:00+00:00,282.773267
3,2019-03-01 12:02:00+00:00,304.049810
4,2019-06-01 00:02:00+00:00,330.086865


## Candidate Continuous Lunar Quantities for F3

The F3 feature family is intended to provide a conventional astronomical
control before the Vedic/Panchang representation is introduced in F4.

This distinction is important for the research question.

If F4 is compared only with a simple Moon-phase feature, an improvement
could merely indicate that the model needed more detailed lunar
astronomical information.

Therefore, F3 will be designed around continuous astronomical quantities
that capture the underlying lunar geometry.

Candidate quantities include:

- Moon-Sun ecliptic longitude difference (continuous lunar phase);
- geocentric apparent lunar ecliptic longitude;
- lunar ecliptic latitude;
- illuminated fraction of the Moon;
- Earth-Moon distance.

The phase angle and ecliptic longitude are circular variables and will
later be represented using sine/cosine components for modelling.

At this stage, no Panchang divisions or sidereal/ayanamsha corrections
are applied.

In [15]:
# ============================================================
# CELL 15 — INSPECT CANDIDATE CONTINUOUS LUNAR QUANTITIES
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Before constructing F3 for the complete SOLETE timeline,
# inspect the continuous astronomical quantities that can
# serve as the non-Vedic lunar control.
#
# We calculate:
#
#     - apparent geocentric lunar ecliptic longitude
#     - apparent geocentric lunar ecliptic latitude
#     - apparent geocentric solar ecliptic longitude
#     - Moon-Sun longitude difference
#     - illuminated fraction
#     - Earth-Moon distance
#
# We also independently reconstruct the Moon-phase angle from
# the ecliptic longitudes and verify that it agrees with
# Skyfield's almanac.moon_phase() result.
#
# No Panchang features are created here.


from skyfield.framelib import ecliptic_frame


# ============================================================
# 1. CALCULATE APPARENT GEOCENTRIC SUN AND MOON POSITIONS
# ============================================================
#
# Earth is the observer, so these quantities are geocentric.
#
# ".observe(...).apparent()" incorporates the apparent
# direction used by Skyfield's astronomical calculations.

earth_at_test_times = earth.at(
    skyfield_test_times
)


moon_apparent_test = (
    earth_at_test_times
    .observe(moon)
    .apparent()
)


sun_apparent_test = (
    earth_at_test_times
    .observe(sun)
    .apparent()
)


# ============================================================
# 2. EXPRESS POSITIONS IN ECLIPTIC COORDINATES
# ============================================================
#
# frame_latlon() returns:
#
#     latitude
#     longitude
#     distance
#
# in the selected ecliptic reference frame.

(
    moon_ecliptic_lat_test,
    moon_ecliptic_lon_test,
    moon_distance_test,
) = moon_apparent_test.frame_latlon(
    ecliptic_frame
)


(
    sun_ecliptic_lat_test,
    sun_ecliptic_lon_test,
    sun_distance_test,
) = sun_apparent_test.frame_latlon(
    ecliptic_frame
)


# ============================================================
# 3. RECONSTRUCT CONTINUOUS MOON PHASE
# ============================================================
#
# Continuous phase:
#
#     Moon longitude - Sun longitude
#
# wrapped into [0°, 360°).

moon_phase_from_longitudes_deg = (
    (
        moon_ecliptic_lon_test.degrees
        -
        sun_ecliptic_lon_test.degrees
    )
    % 360
)


# ============================================================
# 4. VERIFY AGAINST SKYFIELD almanac.moon_phase()
# ============================================================

phase_reconstruction_error_deg = np.abs(
    (
        moon_phase_from_longitudes_deg
        -
        moon_phase_test_deg
        +
        180
    )
    % 360
    -
    180
)


print(
    "Maximum phase reconstruction error [deg]:"
)

print(
    phase_reconstruction_error_deg.max()
)


# ============================================================
# 5. CALCULATE ILLUMINATED FRACTION
# ============================================================
#
# This is the fraction of the Moon's visible disc illuminated
# by the Sun:
#
#     0 -> approximately dark
#     1 -> approximately fully illuminated

moon_illumination_test = (
    moon_apparent_test
    .fraction_illuminated(sun)
)


# ============================================================
# 6. BUILD INSPECTION TABLE
# ============================================================

lunar_geometry_test_results = pd.DataFrame(
    {
        "UTC_timestamp":
            lunar_test_times,

        "moon_phase_angle_deg":
            moon_phase_test_deg,

        "moon_ecliptic_longitude_deg":
            moon_ecliptic_lon_test.degrees,

        "moon_ecliptic_latitude_deg":
            moon_ecliptic_lat_test.degrees,

        "sun_ecliptic_longitude_deg":
            sun_ecliptic_lon_test.degrees,

        "moon_illumination_fraction":
            moon_illumination_test,

        "earth_moon_distance_km":
            moon_distance_test.km,
    }
)


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

print(
    "\nCandidate F3 lunar quantities:"
)

lunar_geometry_test_results

Maximum phase reconstruction error [deg]:
5.684341886080802e-14

Candidate F3 lunar quantities:


,UTC_timestamp,moon_phase_angle_deg,moon_ecliptic_longitude_deg,moon_ecliptic_latitude_deg,sun_ecliptic_longitude_deg,moon_illumination_fraction,earth_moon_distance_km
0,2018-06-01 00:02:00+00:00,206.802147,277.274831,2.552827,70.472685,0.946135,404012.840592
1,2018-09-01 12:02:00+00:00,249.834414,48.839032,-5.063647,159.004618,0.672857,382696.992906
2,2018-12-01 00:02:00+00:00,282.773267,171.499213,4.133658,248.725946,0.390997,374181.153063
3,2019-03-01 12:02:00+00:00,304.049810,284.655756,1.023616,340.605945,0.221064,402524.244415
4,2019-06-01 00:02:00+00:00,330.086865,40.332366,-4.790404,70.245502,0.068473,388252.414873


### Final F3 Feature Design

The F3 feature family is defined as a continuous astronomical control
for the subsequent Vedic/Panchang feature family.

The raw astronomical quantities retained for reproducibility are:

- Moon-Sun phase angle;
- geocentric apparent lunar ecliptic longitude;
- geocentric apparent lunar ecliptic latitude;
- geocentric apparent solar ecliptic longitude;
- illuminated fraction of the Moon;
- Earth-Moon distance.

Circular longitude quantities are not supplied directly to forecasting
models. They are represented using sine/cosine pairs.

The model-ready F3 variables are therefore:

- `moon_phase_sin`
- `moon_phase_cos`
- `moon_longitude_sin`
- `moon_longitude_cos`
- `sun_ecliptic_longitude_sin`
- `sun_ecliptic_longitude_cos`
- `moon_ecliptic_latitude_deg`
- `moon_illumination_fraction`
- `earth_moon_distance_km`

The solar ecliptic longitude is retained even though F2 already contains
local solar geometry. This provides a direct continuous control for the
Sun-Moon ecliptic quantities from which later Panchang variables,
particularly Tithi and Yoga, are constructed.

The illuminated fraction is partly related to lunar phase and may be
statistically redundant. It is retained at the feature-engineering stage
because it provides a physically interpretable scalar representation.
Redundancy will be examined later through model ablation rather than
removed prematurely.

No Tithi, Paksha, Nakshatra, Yoga, Karana, sidereal longitude, or
ayanamsha correction is introduced in F3.

In [16]:
# ============================================================
# CELL 16 — GENERATE FULL F3 CONTINUOUS LUNAR ASTRONOMY
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# We now calculate the validated continuous Sun-Moon
# astronomical quantities for every 5-minute SOLETE row.
#
# Final astronomical time convention:
#
#     SOLETE timestamp + 2 minutes
#
# Final ephemeris:
#
#     JPL DE440s
#
# Coordinate basis:
#
#     geocentric apparent ecliptic coordinates
#
# This cell creates:
#
#     1. df_lunar_geometry
#        raw / audit astronomical quantities
#
#     2. df_lunar_features
#        model-ready F3 variables
#
# No Panchang quantities are created.


# ============================================================
# 1. CONVERT FINAL ASTRONOMICAL REFERENCE TIMES TO SKYFIELD
# ============================================================

skyfield_full_times = ts.from_datetimes(
    astro_reference_times.to_pydatetime()
)


print("Number of astronomical timestamps:")
print(
    len(astro_reference_times)
)


# ============================================================
# 2. CALCULATE APPARENT GEOCENTRIC POSITIONS
# ============================================================

earth_at_full_times = earth.at(
    skyfield_full_times
)


moon_apparent_full = (
    earth_at_full_times
    .observe(moon)
    .apparent()
)


sun_apparent_full = (
    earth_at_full_times
    .observe(sun)
    .apparent()
)


# ============================================================
# 3. CONVERT TO ECLIPTIC COORDINATES
# ============================================================

(
    moon_ecliptic_lat_full,
    moon_ecliptic_lon_full,
    moon_distance_full,
) = moon_apparent_full.frame_latlon(
    ecliptic_frame
)


(
    sun_ecliptic_lat_full,
    sun_ecliptic_lon_full,
    sun_distance_full,
) = sun_apparent_full.frame_latlon(
    ecliptic_frame
)


# ============================================================
# 4. CALCULATE CONTINUOUS MOON-SUN PHASE ANGLE
# ============================================================
#
# We construct this explicitly from the ecliptic longitudes:
#
#     phase = (Moon longitude - Sun longitude) mod 360
#
# This is equivalent to the Skyfield moon_phase definition
# already verified in Cell 15.

moon_phase_angle_full_deg = (
    (
        moon_ecliptic_lon_full.degrees
        -
        sun_ecliptic_lon_full.degrees
    )
    % 360
)


# ============================================================
# 5. CALCULATE LUNAR ILLUMINATION
# ============================================================

moon_illumination_full = (
    moon_apparent_full
    .fraction_illuminated(sun)
)


# ============================================================
# 6. CREATE RAW / AUDIT LUNAR GEOMETRY TABLE
# ============================================================
#
# Raw circular degrees are preserved here for:
#
#     - methodological inspection;
#     - later Panchang derivation;
#     - reproducibility.
#
# They are NOT necessarily supplied directly to models.

df_lunar_geometry = pd.DataFrame(
    {
        "moon_phase_angle_deg":
            moon_phase_angle_full_deg,

        "moon_ecliptic_longitude_deg":
            moon_ecliptic_lon_full.degrees,

        "moon_ecliptic_latitude_deg":
            moon_ecliptic_lat_full.degrees,

        "sun_ecliptic_longitude_deg":
            sun_ecliptic_lon_full.degrees,

        "moon_illumination_fraction":
            moon_illumination_full,

        "earth_moon_distance_km":
            moon_distance_full.km,
    },
    index=df_raw.index,
)


# ============================================================
# 7. CREATE MODEL-READY F3 FEATURE TABLE
# ============================================================

df_lunar_features = pd.DataFrame(
    index=df_raw.index
)


# ------------------------------------------------------------
# Moon-Sun phase angle — circular encoding
# ------------------------------------------------------------

moon_phase_rad = np.deg2rad(
    df_lunar_geometry[
        "moon_phase_angle_deg"
    ]
)

df_lunar_features["moon_phase_sin"] = (
    np.sin(moon_phase_rad)
)

df_lunar_features["moon_phase_cos"] = (
    np.cos(moon_phase_rad)
)


# ------------------------------------------------------------
# Lunar ecliptic longitude — circular encoding
# ------------------------------------------------------------

moon_longitude_rad = np.deg2rad(
    df_lunar_geometry[
        "moon_ecliptic_longitude_deg"
    ]
)

df_lunar_features["moon_longitude_sin"] = (
    np.sin(moon_longitude_rad)
)

df_lunar_features["moon_longitude_cos"] = (
    np.cos(moon_longitude_rad)
)


# ------------------------------------------------------------
# Solar ecliptic longitude — circular encoding
# ------------------------------------------------------------

sun_longitude_rad = np.deg2rad(
    df_lunar_geometry[
        "sun_ecliptic_longitude_deg"
    ]
)

df_lunar_features[
    "sun_ecliptic_longitude_sin"
] = np.sin(
    sun_longitude_rad
)

df_lunar_features[
    "sun_ecliptic_longitude_cos"
] = np.cos(
    sun_longitude_rad
)


# ------------------------------------------------------------
# Non-circular continuous quantities
# ------------------------------------------------------------

df_lunar_features[
    "moon_ecliptic_latitude_deg"
] = df_lunar_geometry[
    "moon_ecliptic_latitude_deg"
]


df_lunar_features[
    "moon_illumination_fraction"
] = df_lunar_geometry[
    "moon_illumination_fraction"
]


df_lunar_features[
    "earth_moon_distance_km"
] = df_lunar_geometry[
    "earth_moon_distance_km"
]


# ============================================================
# 8. BASIC VALIDATION
# ============================================================

print("\nRaw lunar geometry shape:")
print(
    df_lunar_geometry.shape
)


print("\nF3 model-ready feature shape:")
print(
    df_lunar_features.shape
)


print("\nMissing values in F3:")
print(
    df_lunar_features.isna().sum()
)


print("\nRaw lunar geometry ranges:")

for column in df_lunar_geometry.columns:

    print(
        f"{column:32s}: "
        f"{df_lunar_geometry[column].min(): .6f}"
        f" to "
        f"{df_lunar_geometry[column].max(): .6f}"
    )


# ============================================================
# 9. VERIFY CIRCULAR ENCODINGS
# ============================================================

phase_norm = (
    df_lunar_features["moon_phase_sin"] ** 2
    +
    df_lunar_features["moon_phase_cos"] ** 2
)


moon_longitude_norm = (
    df_lunar_features["moon_longitude_sin"] ** 2
    +
    df_lunar_features["moon_longitude_cos"] ** 2
)


sun_longitude_norm = (
    df_lunar_features[
        "sun_ecliptic_longitude_sin"
    ] ** 2
    +
    df_lunar_features[
        "sun_ecliptic_longitude_cos"
    ] ** 2
)


print(
    "\nMaximum phase circular-norm error:"
)
print(
    np.abs(
        phase_norm - 1
    ).max()
)


print(
    "\nMaximum Moon-longitude circular-norm error:"
)
print(
    np.abs(
        moon_longitude_norm - 1
    ).max()
)


print(
    "\nMaximum Sun-longitude circular-norm error:"
)
print(
    np.abs(
        sun_longitude_norm - 1
    ).max()
)


# ============================================================
# 10. DISPLAY SAMPLE
# ============================================================

print(
    "\nSample F3 lunar astronomical features:"
)

df_lunar_features.head(12)

Number of astronomical timestamps:
131617

Raw lunar geometry shape:
(131617, 6)

F3 model-ready feature shape:
(131617, 9)

Missing values in F3:
moon_phase_sin                0
moon_phase_cos                0
moon_longitude_sin            0
moon_longitude_cos            0
sun_ecliptic_longitude_sin    0
sun_ecliptic_longitude_cos    0
moon_ecliptic_latitude_deg    0
moon_illumination_fraction    0
earth_moon_distance_km        0
dtype: int64

Raw lunar geometry ranges:
moon_phase_angle_deg            :  0.000493 to  359.999646
moon_ecliptic_longitude_deg     :  0.002773 to  359.998512
moon_ecliptic_latitude_deg      : -5.301922 to  5.293951
sun_ecliptic_longitude_deg      :  0.002462 to  359.999014
moon_illumination_fraction      :  0.000031 to  0.999999
earth_moon_distance_km          :  356762.768577 to  406559.424079

Maximum phase circular-norm error:
2.220446049250313e-16

Maximum Moon-longitude circular-norm error:
2.220446049250313e-16

Maximum Sun-longitude circular-norm erro

,moon_phase_sin,moon_phase_cos,moon_longitude_sin,moon_longitude_cos,sun_ecliptic_longitude_sin,sun_ecliptic_longitude_cos,moon_ecliptic_latitude_deg,moon_illumination_fraction,earth_moon_distance_km
Timestamp,,,,,,,,,
2018-06-01 00:00:00+00:00,-0.450911,-0.892569,-0.991950,0.126629,0.942482,0.334256,2.552827,0.946135,404012.840592
2018-06-01 00:05:00+00:00,-0.451504,-0.892269,-0.991858,0.127346,0.942502,0.334201,2.549630,0.945987,404017.839327
2018-06-01 00:10:00+00:00,-0.452097,-0.891969,-0.991766,0.128062,0.942521,0.334147,2.546432,0.945839,404022.829368
2018-06-01 00:15:00+00:00,-0.452690,-0.891668,-0.991673,0.128779,0.942540,0.334092,2.543233,0.945691,404027.810709
2018-06-01 00:20:00+00:00,-0.453282,-0.891367,-0.991580,0.129496,0.942560,0.334037,2.540033,0.945542,404032.783341
2018-06-01 00:25:00+00:00,-0.453874,-0.891066,-0.991486,0.130212,0.942579,0.333983,2.536831,0.945394,404037.747260
2018-06-01 00:30:00+00:00,-0.454466,-0.890764,-0.991392,0.130928,0.942599,0.333928,2.533628,0.945245,404042.702457
2018-06-01 00:35:00+00:00,-0.455058,-0.890462,-0.991297,0.131645,0.942618,0.333873,2.530424,0.945096,404047.648926
2018-06-01 00:40:00+00:00,-0.455650,-0.890159,-0.991202,0.132361,0.942637,0.333818,2.527218,0.944946,404052.586660


## F3 Continuous Lunar-Astronomy Outcome

Continuous lunar astronomical features were successfully generated for
all **131,617** observations in the 5-minute SOLETE timeline.

The calculation uses:

- the verified UTC timeline;
- astronomical reference time `t + 2 minutes`;
- Skyfield `1.55`;
- JPL `DE440s`;
- apparent geocentric ecliptic coordinates.

### Raw astronomical quantities

The reproducibility/audit table contains:

- Moon-Sun phase angle;
- lunar ecliptic longitude;
- lunar ecliptic latitude;
- solar ecliptic longitude;
- lunar illuminated fraction;
- Earth-Moon distance.

All quantities are complete with no missing observations.

Across the dataset:

- lunar phase covers essentially the complete `0–360°` cycle;
- lunar longitude covers essentially the complete `0–360°` cycle;
- solar ecliptic longitude covers essentially the complete `0–360°` cycle;
- lunar ecliptic latitude ranges approximately from `-5.30°` to `+5.29°`;
- illuminated fraction ranges approximately from `0` to `1`;
- Earth-Moon distance ranges approximately from `356,763 km`
  to `406,559 km`.

### Model-ready F3 variables

Circular astronomical quantities are represented using sine/cosine
pairs:

- `moon_phase_sin`
- `moon_phase_cos`
- `moon_longitude_sin`
- `moon_longitude_cos`
- `sun_ecliptic_longitude_sin`
- `sun_ecliptic_longitude_cos`

The remaining continuous variables are:

- `moon_ecliptic_latitude_deg`
- `moon_illumination_fraction`
- `earth_moon_distance_km`

All circular encodings satisfy the expected unit-circle identity to
floating-point precision.

### Methodological role of F3

F3 acts as the continuous astronomical control for the later
Vedic/Panchang feature family.

This separation allows the forecasting experiments to distinguish
between:

1. improvements arising from ordinary continuous Sun-Moon astronomy;
2. improvements arising specifically from the Vedic/Panchang
   representation of that astronomical geometry.

No Tithi, Paksha, Nakshatra, Yoga, Karana, Masa, sidereal longitude, or
ayanamsha correction is introduced in F3.